# Chapter 8 &mdash; Anatomy of the RE-to-NFA Converter: a Mini-Compiler

**Concept 5 of the Chapter 8 decomposition:** *Anatomy of the RE-to-NFA Converter: a Mini-Compiler*

`re2nfa` is a lexer plus a parser whose production rules assemble NFA fragments.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-RE-Converter-Anatomy/Concept-RE-Converter-Anatomy.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


`re2nfa` is a small but complete **compiler**:

* a **lexer** (`lex()`) turning the RE text into tokens &mdash; `t_PLUS`, `t_STAR`,
  `t_LPAREN`, `t_RPAREN`, `t_EPS`, `t_STR`;
* a **parser** (`yacc()`) with one production per RE operator, whose **semantic
  action** is a fragment constructor from Concept 2.

The grammar layers encode precedence: `expression` handles `+`, `catexp` handles
juxtaposition, `ordyexp` handles `*` and parentheses. Union is loosest because it sits
at the top.

Reading this converter teaches you how *any* small language is implemented &mdash; and
Chapter 11's grammars are the same idea from the other side.

## 2. Definitions

### The tokens the lexer recognises

In [ ]:
import jove.Def_RE2NFA as C
for t in ['t_PLUS', 't_STAR', 't_LPAREN', 't_RPAREN', 't_EPS', 't_STR']:
    print("%-10s = %r" % (t, getattr(C, t)))
print("\ntoken list :", C.tokens)

### The grammar productions, read off the parser functions

In [ ]:
import inspect
def productions():
    for name in sorted(n for n in dir(C) if n.startswith('p_') and n != 'p_error'):
        doc = (getattr(C, name).__doc__ or '').strip()
        print("%-26s %s" % (name, doc))

## 3. Tests

The grammar, one production per line.

In [ ]:
productions()

The three layers **are** the precedence: `+` at the top, `*` at the bottom.

In [ ]:
print("expression : expression PLUS catexp     <- union, loosest")
print("catexp     : catexp ordyexp             <- concatenation")
print("ordyexp    : ordyexp STAR | ( expr )    <- star and grouping, tightest")
print()
print("so 0+1*  parses as 0 + (1*) :",
      sorted({s for s in ['', '0', '1', '11', '01'] if accepts_nfa(re2nfa('0+1*'), s)}))

Each production's **action** builds a fragment &mdash; the compiler's code generator.

In [ ]:
src = inspect.getsource(C.p_expression_plus)
print(src.strip()[:400])

The whole pipeline, end to end, on one expression.

In [ ]:
r = "(0+1)*1"
N = re2nfa(r)
print("RE     :", r)
print("tokens : lexed by lex(), parsed by yacc()")
print("NFA    : |Q| = %d, Q0 = %s, F = %s" % (len(N["Q"]), sorted(N["Q0"]), sorted(N["F"])))
assert accepts_nfa(N, '1') and accepts_nfa(N, '0001') and not accepts_nfa(N, '10')

A syntax error is reported by `p_error`, not by a crash deep inside.

In [ ]:
try:
    re2nfa("(0+1")
    print("no error raised")
except Exception as e:
    print("parse failure :", type(e).__name__, str(e)[:60])

## 4. Animation

The NFA the mini-compiler emitted.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(re2nfa('(0+1)*1'), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Add a `?` (optional) operator. Which production and which constructor?
2. Why is `catexp` a separate layer rather than a precedence declaration?
3. Compare this converter with the `md2mc` parser. What do they share?

In [ ]:
# Your work for the exercises above.